<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/xGBoostModelreviewed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade pandas numpy scikit-learn xgboost

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score,classification_report
from xgboost import XGBClassifier
import pickle
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

In [6]:
import os

print(os.path.getsize("sparse_dict.pkl"))

735277681


In [7]:
#Load the sparse dataset
import pickle
with open("sparse_dict.pkl", "rb") as f:
  sparse = pickle.load(f)


In [8]:
print(sparse.keys())

dict_keys(['sparse_train', 'sparse_val', 'sparse_test'])


In [9]:
for name, hts in sparse.items():
  print(name, hts.shape)

sparse_train (1271440, 56)
sparse_val (423814, 56)
sparse_test (423814, 56)


In [10]:
cols_to_drop = ["EMR", "SDP", "County","PatientPk","MFL_Code","SiteCode","Latitude","Longitude","Facility_Name", "SubCounty",
                "ClientTestedAs","CoupleDiscordant","VisitDate","NumberOfPartners"]

for key in sparse.keys():
  sparse[key] = sparse[key].drop(columns=cols_to_drop, errors="ignore")


In [11]:
#Check dtypes
for name, df in sparse.items():
    print(f"\n{name}")
    print(df.dtypes.value_counts())



sparse_train
object     41
float64     1
Name: count, dtype: int64

sparse_val
object     41
float64     1
Name: count, dtype: int64

sparse_test
object     41
float64     1
Name: count, dtype: int64


Prepare train, validation and test sets

In [12]:
#define features and target variable
X_train = sparse["sparse_train"].drop(columns=["FinalTestResult"])
y_train = sparse["sparse_train"]["FinalTestResult"]

X_val = sparse["sparse_val"].drop(columns=["FinalTestResult"])
y_val = sparse["sparse_val"]["FinalTestResult"]

X_test = sparse["sparse_test"].drop(columns=["FinalTestResult"])
y_test = sparse["sparse_test"]["FinalTestResult"]

Encode categorical variables

In [13]:
dtype_summary = pd.DataFrame({
    "Variable": X_train.columns,
    "Datatype": X_train.dtypes.astype(str)
})

dtype_summary = dtype_summary.sort_values("Datatype")

dtype_summary.head(100)

,Variable,Datatype
Age,Age,float64
PopulationType,PopulationType,object
KeyPopulation,KeyPopulation,object
PriorityPopulation,PriorityPopulation,object
IsHealthWorker,IsHealthWorker,object
TestedHIVBefore,TestedHIVBefore,object
ResultOfHIV,ResultOfHIV,object
EverHadSex,EverHadSex,object
SexuallyActive,SexuallyActive,object
NewPartner,NewPartner,object


In [14]:
for col in X_train.columns:
    print(col, X_train[col].nunique())


PopulationType 3
KeyPopulation 5
PriorityPopulation 1
IsHealthWorker 3
TestedHIVBefore 2
ResultOfHIV 2
EverHadSex 3
SexuallyActive 5
NewPartner 4
PartnerHIVStatus 5
MultiplePartners 3
AlcoholSex 5
MoneySex 4
CondomBurst 4
UnknownStatusPartner 4
KnownStatusPartner 4
Pregnant 4
BreastfeedingMother 4
ExperiencedViolenceScreening 4
CurrentlyOnPrep 3
TraditionalProcedures 2
MothersStatus 4
ResultOfHIVSelf 2
ScreenedTB 3
TBStatus 4
ReceivedServices 9
ViolenceScreeningType 9
Sex 2
MaritalStatus 7
PatientDisabled 2
EverTestedForHiv 2
MonthsSinceLastTest 5
PriorityPopulationType 5
Age 99
GBVSexual 3
GBVPhysical 3
GBVEmotional 3
ReceivedPREP 2
ReceivedTB 2
ReceivedSTI 2
dayofweek 7


In [15]:
# One-hot encode categorical features into sparse matrices
from scipy import sparse as scipy_sparse
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns

encoder = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), categorical_cols)
    ],
    remainder="passthrough",
    sparse_threshold=1.0,
)

X_train_enc = scipy_sparse.csr_matrix(encoder.fit_transform(X_train))
X_val_enc = scipy_sparse.csr_matrix(encoder.transform(X_val))
X_test_enc = scipy_sparse.csr_matrix(encoder.transform(X_test))
encoded_feature_names = encoder.get_feature_names_out()

In [16]:
# Verify that all splits have the same encoded features
print(X_train_enc.shape[1] == X_val_enc.shape[1])
print(X_train_enc.shape[1] == X_test_enc.shape[1])

True
True


In [17]:
# Preview a small dense slice without expanding the full sparse matrix
pd.DataFrame(
    X_train_enc[:5, :10].toarray(),
    columns=encoded_feature_names[:10],
)

,categorical__PopulationType_GP,categorical__PopulationType_KP,categorical__PopulationType_PRIORITY,categorical__KeyPopulation_FSW,categorical__KeyPopulation_MSM,categorical__KeyPopulation_NR,categorical__KeyPopulation_PRISONER,categorical__KeyPopulation_PWID,categorical__KeyPopulation_nan,categorical__PriorityPopulation_NR
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [18]:
print("shape:", X_train_enc.shape)
print("dtype:", X_train_enc.dtype)
print("stored nonzeros:", X_train_enc.nnz)

shape: (1271440, 183)
dtype: float64
stored nonzeros: 52129040


In [19]:
density = X_train_enc.nnz / (X_train_enc.shape[0] * X_train_enc.shape[1])
print(f"Sparse matrix density: {density:.6f}")

Sparse matrix density: 0.224044


Encode target variable

In [20]:
#Check the output, if not in numericals, encode it
print(y_train.value_counts())

FinalTestResult
NEGATIVE    1239327
POSITIVE      32113
Name: count, dtype: int64


In [21]:
# Encode target labels using the training-set mapping
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

In [22]:
#Calculate this if positivity is much fewer than negatives: This is because HIV positivity is usually imbalanced
neg = (y_train_enc == 0).sum()
pos = (y_train_enc == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

38.592688319372215


Train XGBoost

In [ ]:
# CHANGED: Define the hyperparameter grid and retain the best model/results.
from itertools import product
import time
import pandas as pd

max_depths = [5, 7]
learning_rates = [0.05, 0.1]
scale_pos_weights = [1, 10]
subsamples = [0.8, 1.0]

# CHANGED: Store every validation score instead of only overwriting xgb_model.
results = []
best_model = None
best_score = float("-inf")
best_params = None

# CHANGED: Track progress and estimate the remaining runtime.
configurations = list(product(
    max_depths,
    learning_rates,
    scale_pos_weights,
    subsamples
))
total_runs = len(configurations)
search_start = time.perf_counter()

for run_number, (max_depth, learning_rate, scale_pos_weight, subsample) in enumerate(
    configurations, start=1
):
    print(
        f"Starting {run_number}/{total_runs} ({run_number / total_runs:.1%}): "
        f"depth={max_depth}, lr={learning_rate}, spw={scale_pos_weight}, "
        f"subsample={subsample}",
        flush=True,
    )

    xgb_model = XGBClassifier(
        n_estimators=1000,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        subsample=subsample,
        random_state=2231,
        eval_metric="aucpr",
        early_stopping_rounds=50,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train_enc,
        y_train_enc,
        eval_set=[(X_val_enc, y_val_enc)],
        verbose=False
    )

    score = xgb_model.best_score
    params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "scale_pos_weight": scale_pos_weight,
        "subsample": subsample,
    }
    results.append({
        **params,
        "validation_aucpr": score,
        "best_iteration": xgb_model.best_iteration,
    })

    # CHANGED: Select the model using validation AUPRC, matching eval_metric.
    if score > best_score:
        best_score = score
        best_model = xgb_model
        best_params = params

    elapsed = time.perf_counter() - search_start
    average_time = elapsed / run_number
    eta_seconds = average_time * (total_runs - run_number)
    print(
        f"Completed {run_number}/{total_runs}; "
        f"AUPRC={score:.5f}; "
        f"elapsed={elapsed / 60:.1f} min; "
        f"ETA={eta_seconds / 60:.1f} min",
        flush=True,
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("validation_aucpr", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv("xgb_hyperparameter_results.csv", index=False)

expected_runs = len(configurations)
print(f"Completed {len(results)} of {expected_runs} configurations")
print("Best parameters:", best_params)
print("Best validation AUPRC:", best_score)
print("Best iteration:", best_model.best_iteration)
results_df.head(10)

Starting 1/16 (6.2%): depth=5, lr=0.05, spw=1, subsample=0.8
Completed 1/16; AUPRC=0.21757; elapsed=49.0 min; ETA=735.2 min
Starting 2/16 (12.5%): depth=5, lr=0.05, spw=1, subsample=1.0


In [ ]:
# Save
joblib.dump(xgb_model,
            '/content/drive/MyDrive/xgb_model.pkl')
# Save
joblib.dump(best_model,
            '/content/drive/MyDrive/best_model.pkl')
from google.colab import files

files.download("xgb_model.pkl")

from google.colab import files

files.download("best_model.pkl")

Checking the maximum threshhold at maximum precision at 90%  and 95% sensitivity

In [ ]:
for target in [0.90, 0.95]:

    eligible = pr_df[pr_df["recall"] >= target]

    best_row = eligible.loc[eligible["precision"].idxmax()]

    print(f"\nRecall ≥ {target:.0%}")
    print(f"Recall: {best_row['recall']:.4f}")
    print(f"Precision: {best_row['precision']:.4f}")
    print(f"Threshold: {best_row['threshold']:.6f}")
    print(f"Precision lift: {best_row['precision']/prevalence:.2f}x")

In [ ]:
for target in [0.90, 0.95]:
    subset = pr_df[pr_df["recall"] >= target]

    best = subset.loc[subset["precision"].idxmax()]

    print(best[["recall","precision","threshold"]])

Validate the model at 90%

In [ ]:
# Use the threshold selected earlier
threshold_90 = best_row["threshold"]

# Validation probabilities
y_val_prob = best_model.predict_proba(X_val_enc)[:, 1]

# Validation predictions
y_val_pred = (y_val_prob >= threshold_90).astype(int)

print(classification_report(
    y_val_enc,
    y_val_pred,
    digits=4
))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Validation probabilities
y_val_prob = best_model.predict_proba(X_val_enc)[:, 1]

# Compute PR curve
val_precision, val_recall, val_thresholds = precision_recall_curve(
    y_val_enc,
    y_val_prob
)

# Average Precision (AUPRC)
val_ap = average_precision_score(y_val_enc, y_val_prob)

# Prevalence baseline
val_prevalence = y_val_enc.mean()

plt.figure(figsize=(8,6))

plt.plot(
    val_recall,
    val_precision,
    linewidth=2,
    label=f"Validation PR Curve (AP={val_ap:.3f})"
)

# Baseline prevalence
plt.axhline(
    val_prevalence,
    color="red",
    linestyle="--",
    label=f"Prevalence={val_prevalence:.3%}"
)

plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Validation Precision-Recall Curve")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val_enc, y_val_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot(cmap="Blues")
plt.title("Validation Set Confusion Matrix")
plt.show()

Validate the model at 95%

Test the model at 90% sensitivity

In [ ]:
eligible = pr_df[pr_df["recall"] >= 0.90]

best_row = eligible.loc[
    eligible["precision"].idxmax()
]

best_row

In [ ]:
threshold_90 = best_row["threshold"]

y_pred_90 = (
    best_model.predict_proba(X_test_enc)[:,1]
    >= threshold_90
).astype(int)

print(classification_report(
    y_test_enc,
    y_pred_90,
    digits=4
))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Probabilities
y_prob = best_model.predict_proba(X_test_enc)[:, 1]

# PR curve
precisions, recalls, thresholds = precision_recall_curve(
    y_test_enc,
    y_prob
)

ap = average_precision_score(y_test_enc, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, linewidth=2,
         label=f'PR Curve (AP = {ap:.3f})')

# Mark the 90% recall operating point
plt.scatter(
    0.90,
    0.0491,
    color='red',
    s=100,
    label='90% Recall Threshold'
)

plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_enc, y_pred_90)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax)

plt.title(
    f"Confusion Matrix\nRecall={0.90:.2%}, Precision={0.0491:.2%}"
)
plt.show()

Test the model at 95% sensitivity

In [ ]:
best_row_95 = (
    pr_df[pr_df["recall"] >= 0.95]
    .sort_values("precision", ascending=False)
    .iloc[0]
)

threshold_95 = best_row_95["threshold"]

y_pred_95 = (
    best_model.predict_proba(X_test_enc)[:, 1]
    >= threshold_95
).astype(int)

print(classification_report(
    y_test_enc,
    y_pred_95,
    digits=4
))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Probabilities
y_prob = best_model.predict_proba(X_test_enc)[:, 1]

# PR curve
precisions, recalls, thresholds = precision_recall_curve(
    y_test_enc,
    y_pred_95
)

ap = average_precision_score(y_test_enc, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, linewidth=2,
         label=f'PR Curve (AP = {ap:.3f})')

# Mark the 90% recall operating point
plt.scatter(
    0.90,
    0.0491,
    color='red',
    s=100,
    label='90% Recall Threshold'
)

plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_enc, y_pred_95)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax)

plt.title(
    f"Confusion Matrix\nRecall={0.90:.2%}, Precision={0.0491:.2%}"
)
plt.show()